# Comparativa a ciegas de TTS open source vs ElevenLabs (ep12_es, voz Nebu)

Genera los mismos 6 planos de ep12_es con **Chatterbox Multilingual** (MIT) y **Qwen3-TTS 1.7B** (Apache 2.0) clonando tu voz. Después te los pone **a ciegas** al lado de la toma real de ElevenLabs. IndexTTS-2 es opcional (no admite español oficialmente).

**Antes de empezar (una vez):**
1. En Kaggle, *Settings → Phone verification* (sin eso no hay GPU ni Internet).
2. Sube como *Dataset* la carpeta `dataset/` de este repo **y añade tu audio de referencia** `nebu_ref.wav` (o .mp3/.m4a): 8–15 s, solo tu voz, sin música, una frase completa con algo de emoción. Opcional: `nebu_ref.txt` con lo que dices exactamente.
3. En el notebook: *Add Input* → tu dataset. *Settings → Accelerator →* **GPU T4 x2** (no P100). *Internet* → **On**.
4. *Run All*. Tarda unos 20–30 min la primera vez (instalación + descarga de modelos).

Cada motor se instala en su propio entorno (`uv`), así no se rompen entre sí ni con el PyTorch de Kaggle.

In [ ]:
# ===== CONFIGURACIÓN =====
RUN_CHATTERBOX = True
RUN_QWEN       = True
RUN_INDEXTTS   = False   # opcional: solo inglés/chino oficialmente, probablemente pronuncie mal el español
TAKES          = 2       # tomas por plano y motor (eliges la mejor, como harías con ElevenLabs)
TEMPO          = 1.2     # ElevenLabs se aceleró x1.2 en ep12; se aplica igual a los demás para comparar justo
REF_TEXT       = None    # pon aquí la transcripción exacta si no subes nebu_ref.txt (si no, la hace Whisper)
REF_MAX_SECONDS = 15     # se recorta la referencia si es más larga

import glob, os, json, subprocess, shutil
W = "/kaggle/working"; OUT = f"{W}/out"; os.makedirs(OUT, exist_ok=True)
shots_json = glob.glob("/kaggle/input/**/shots.json", recursive=True)
assert shots_json, "No encuentro shots.json: añade el dataset con Add Input"
DATA = os.path.dirname(shots_json[0])
refs = [p for ext in ("wav", "mp3", "m4a", "flac", "ogg") for p in glob.glob(f"/kaggle/input/**/nebu_ref.{ext}", recursive=True)]
assert refs, "Falta nebu_ref.wav (o .mp3/.m4a) en el dataset"
print("dataset:", DATA, "| referencia:", refs[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Preparar la referencia (24 kHz mono, volumen normalizado) y su transcripción

In [ ]:
REF = f"{W}/nebu_ref.wav"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", refs[0], "-t", str(REF_MAX_SECONDS),
                "-af", "loudnorm=I=-18:TP=-1.5", "-ar", "24000", "-ac", "1", REF], check=True)
txt = glob.glob("/kaggle/input/**/nebu_ref.txt", recursive=True)
if REF_TEXT is None and txt:
    REF_TEXT = open(txt[0], encoding="utf-8").read().strip()
if REF_TEXT is None:
    import torch
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
                   torch_dtype=torch.float16, device=0)
    REF_TEXT = asr(REF, generate_kwargs={"language": "spanish", "task": "transcribe"})["text"].strip()
    del asr; torch.cuda.empty_cache()
print("Transcripción de la referencia (corrígela en REF_TEXT si está mal):\n", REF_TEXT)

cfg = json.load(open(f"{DATA}/shots.json", encoding="utf-8"))
cfg.update(ref_wav=REF, ref_text=REF_TEXT, out_dir=OUT, takes=TAKES)
json.dump(cfg, open(f"{W}/config.json", "w"), ensure_ascii=False, indent=1)
from IPython.display import Audio, display; display(Audio(REF))

## 2. Instalar `uv` (gestor de entornos rápido)

In [ ]:
!pip install -q uv
!uv --version

## 3. Chatterbox Multilingual (MIT)
`exaggeration` sube la emoción y `cfg_weight` bajo da un ritmo más expresivo. Los valores de cada plano están en `shots.json`.

In [ ]:
%%writefile /kaggle/working/run_chatterbox.py
import json, sys, os, inspect, torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    m = ChatterboxMultilingualTTS.from_pretrained(device=dev, t3_model="v3")
except TypeError:
    m = ChatterboxMultilingualTTS.from_pretrained(device=dev)
accepted = inspect.signature(m.generate).parameters
out = os.path.join(cfg["out_dir"], "chatterbox"); os.makedirs(out, exist_ok=True)
for s in cfg["shots"]:
    kw = dict(language_id="es", audio_prompt_path=cfg["ref_wav"],
              exaggeration=s["cb_exaggeration"], cfg_weight=s["cb_cfg_weight"])
    kw = {k: v for k, v in kw.items() if k in accepted}
    for t in range(cfg["takes"]):
        wav = m.generate(s["text"], **kw)
        ta.save(f"{out}/shot_{s['id']:02d}_t{t+1}.wav", wav.cpu(), m.sr)
        print("chatterbox", s["id"], "toma", t + 1, "ok", flush=True)

In [ ]:
if RUN_CHATTERBOX:
    !uv venv -q --python 3.11 /tmp/venv_cb
    !uv pip install -q --python /tmp/venv_cb/bin/python chatterbox-tts
    !/tmp/venv_cb/bin/python /kaggle/working/run_chatterbox.py /kaggle/working/config.json

## 4. Qwen3-TTS 1.7B Base (Apache 2.0)
⚠️ Al clonar una voz, Qwen3-TTS **no admite instrucciones de emoción**: solo las saca del texto y la puntuación, y se parece a la emoción de tu audio de referencia. Prueba también una referencia más expresiva.

In [ ]:
%%writefile /kaggle/working/run_qwen.py
import json, sys, os, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32  # la T4 no tiene bf16 real
m = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
                                  dtype=dtype, attn_implementation="sdpa")
lang = "Auto"
try:
    lang = next((l for l in m.get_supported_languages() if "spanish" in str(l).lower()), "Auto")
except Exception:
    pass
print("idioma:", lang)
prompt = m.create_voice_clone_prompt(ref_audio=cfg["ref_wav"], ref_text=cfg["ref_text"], x_vector_only_mode=False)
out = os.path.join(cfg["out_dir"], "qwen3"); os.makedirs(out, exist_ok=True)
for s in cfg["shots"]:
    for t in range(cfg["takes"]):
        wavs, sr = m.generate_voice_clone(text=s["text"], language=lang, voice_clone_prompt=prompt)
        sf.write(f"{out}/shot_{s['id']:02d}_t{t+1}.wav", wavs[0], sr)
        print("qwen3", s["id"], "toma", t + 1, "ok", flush=True)

In [ ]:
if RUN_QWEN:
    !uv venv -q --python 3.12 /tmp/venv_qwen
    !uv pip install -q --python /tmp/venv_qwen/bin/python qwen-tts torch torchaudio soundfile
    !/tmp/venv_qwen/bin/python /kaggle/working/run_qwen.py /kaggle/working/config.json

## 5. (Opcional) IndexTTS-2: emoción separada de la voz
Se controla con un vector `[alegre, enfadado, triste, miedo, asco, melancólico, sorprendido, calma]`. Su licencia (bilibili) pide contactarles para uso comercial.

In [ ]:
%%writefile /kaggle/working/run_indextts.py
import json, sys, os
from indextts.infer_v2 import IndexTTS2
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
tts = IndexTTS2(cfg_path="checkpoints_2/config.yaml", model_dir="checkpoints_2", use_fp16=True,
                use_cuda_kernel=False, use_deepspeed=False)
out = os.path.join(cfg["out_dir"], "indextts2"); os.makedirs(out, exist_ok=True)
for s in cfg["shots"]:
    for t in range(cfg["takes"]):
        tts.infer(spk_audio_prompt=cfg["ref_wav"], text=s["text"], emo_vector=s["emo_vector"],
                  use_random=False, output_path=f"{out}/shot_{s['id']:02d}_t{t+1}.wav", verbose=False)
        print("indextts2", s["id"], "toma", t + 1, "ok", flush=True)

In [ ]:
if RUN_INDEXTTS:
    !git clone -q https://github.com/index-tts/index-tts.git /tmp/index-tts
    !cd /tmp/index-tts && uv sync -q
    !cd /tmp/index-tts && uvx --from huggingface_hub hf download IndexTeam/IndexTTS-2 --local-dir=checkpoints_2
    !cd /tmp/index-tts && PYTHONPATH=. uv run python /kaggle/working/run_indextts.py /kaggle/working/config.json

## 6. Mismo tempo y volumen para todos (y ElevenLabs como referencia)

In [ ]:
def norm(src, dst, tempo):
    af = (f"atempo={tempo}," if tempo != 1 else "") + "loudnorm=I=-16:TP=-1.5"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-af", af, "-ar", "24000", "-ac", "1", dst], check=True)

FINAL = f"{W}/final"; shutil.rmtree(FINAL, ignore_errors=True)
engines = [e for e in ("chatterbox", "qwen3", "indextts2") if os.path.isdir(f"{OUT}/{e}")]
for e in engines:
    os.makedirs(f"{FINAL}/{e}", exist_ok=True)
    for f in sorted(glob.glob(f"{OUT}/{e}/*.wav")):
        norm(f, f"{FINAL}/{e}/{os.path.basename(f)}", TEMPO)
os.makedirs(f"{FINAL}/elevenlabs", exist_ok=True)
for f in sorted(glob.glob(f"{DATA}/elevenlabs/*.wav")):
    norm(f, f"{FINAL}/elevenlabs/{os.path.basename(f)}", 1)   # ya tiene el tempo aplicado
print("motores:", engines + ["elevenlabs"])

## 7. Escucha a ciegas
En cada plano, puntúa A/B/C… del 1 al 5 en **naturalidad**, **parecido a tu voz** y **emoción**. Anótalo en la celda de abajo **antes** de ver la solución. De cada motor sale su primera toma; el resto de tomas está en `final/`.

In [ ]:
import random
from IPython.display import Audio, display, HTML
random.seed(12)
KEY = {}
for s in cfg["shots"]:
    cands = [("elevenlabs", f"{FINAL}/elevenlabs/shot_{s['id']:02d}.wav")]
    cands += [(e, f"{FINAL}/{e}/shot_{s['id']:02d}_t1.wav") for e in engines]
    cands = [c for c in cands if os.path.exists(c[1])]
    random.shuffle(cands)
    display(HTML(f"<h3>Plano {s['id']} · <i>{s['tag']}</i></h3><p>{s['text']}</p>"))
    KEY[s["id"]] = {}
    for letter, (eng, path) in zip("ABCDE", cands):
        KEY[s["id"]][letter] = eng
        display(HTML(f"<b>{letter}</b>")); display(Audio(path))

In [ ]:
# Tus notas (naturalidad, parecido, emoción), por ejemplo  NOTAS = {1: {"A": (4,5,3), "B": (3,3,2)}, ...}
NOTAS = {}

In [ ]:
# SOLUCIÓN: ejecútala solo cuando hayas puntuado
from collections import defaultdict
for sid, k in KEY.items(): print(sid, k)
tot = defaultdict(list)
for sid, d in NOTAS.items():
    for letter, n in d.items(): tot[KEY[sid][letter]].append(sum(n) / len(n))
for e, v in sorted(tot.items(), key=lambda x: -sum(x[1]) / len(x[1])): print(f"{e:12s} {sum(v)/len(v):.2f}/5")

## 8. Descargar todo (botón *Output* del panel derecho, o este zip)

In [ ]:
shutil.make_archive(f"{W}/tts_bakeoff_ep12", "zip", FINAL)
print("Listo:", f"{W}/tts_bakeoff_ep12.zip")